# SANS analysis, part II: priors and statistics

You have now [simulated](./../3-mcstas/mcstas-sans.ipynb), [reduced](./../4-reduction/reduction-sans.ipynb), and carried out [routine analysis](./6a-analysis-sans.ipynb) of your SANS data.

In this notebook, we begin looking at model fitting [probabilistically](./2-prob_data.ipynb), including prior knowledge about the system or parameters under study in the analysis.

We pick up directly where the [previous notebook](./6a-analysis-sans.ipynb) left off, so we start by loading the same dataset again.


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="bumps")
warnings.filterwarnings("ignore", category=UserWarning, message="Message serialization failed")

In [ ]:
import utils
from sans_fitter import SANSFitter

In [ ]:
filename = "../4-reduction/sans_iofq.dat"

⚠️ **If you did not complete the SANS data reduction yesterday**,
you can use some pre-prepared data by uncommenting and running the cell below:

In [ ]:
# filename = utils.fetch_data("4-reduction/sans_iofq.dat")

Starting with the same dataset, set a sphere model and fetch the parameters, as before.

In [ ]:
fitter = SANSFitter()
fitter.load_data(filename)
fitter.set_model("sphere")
fitter.get_params()

By default, some parameters are set but nothing is allowed to vary. Obviously, this will constrain our fitting algorithm excessively. 

In the [previous notebook](./6a-analysis-sans.ipynb), we included prior knowledge in the analysis through the use of bounded parameters. 

Bounded parameters cannot have values less than some lower bound (Min) or greater than some upper bound (Max), as the probability of the parameters having these values is zero. 
For example, if the parameter `b` from a quadratic model has bounds of 0 and 10, then there is an equal probability that the value of `b` can be anything in between 0 and 10, and a probability of 0 outside those bounds, i.e., it has a uniform prior probability distribution.

### Exercise 6: Towards Bayesian analysis

In this exercise we will repeat the sphere fit from Part I, but treat it probabilistically: the bounds we set act as uniform priors, and instead of a single best-fit value we will obtain a *posterior probability distribution* for each varied parameter.

The exercise proceeds in three steps:

1. Set sensible initial values and bounds (priors) for the sphere model, choose which parameters should vary, and confirm the input.
2. Sample the posterior distribution using the `DREAM` Markov chain Monte Carlo algorithm.
3. Inspect the results with a series of diagnostic plots, and consider what they tell you about the parameters and their uncertainties.

Start with step 1, using the same values and bounds as before:

In [ ]:
fitter.set_param("sld", value=3, min=1, max=30, vary=False)
fitter.set_param("sld_solvent", value=6, min=1, max=30, vary=False)

fitter.set_param("radius", value=80, min=10, max=300, vary=True)
fitter.set_param("scale", value=1.4e-7, min=0, max=1, vary=True)
fitter.set_param("background", value=0.1, min=0, max=1, vary=True)

fitter.get_params()

Great: Now the model is set up, let's introduce some new syntax for $Bayesian$ analysis using `sans-fitter`. 

`SasView`, and hence `sans-fitter`, have a number of optimisation algorithms built-in. One of these is `DREAM`, a population based algorithm. 

`DREAM` is relatively slow; it follows a differential evolution-like process but sometimes keeps individuals which get worse with the evolution and allows these to progress as a Markov chain which converges on the equilibrium distribution, where the chain draws randomly from the posterior distribution. 

Therefore, we can use the `DREAM` fitting algorithm to determine parameter uncertainties from our fitting process. 

[More can be read about `DREAM` in the associated publication](https://www.degruyterbrill.com/document/doi/10.1515/IJNSNS.2009.10.3.273/html), or in the [SasView documentation](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/optimizer.html#fit-dream). 

We will begin by setting a few parameters: `samples` (number of points to be drawn from the Markov chain) and `burn` (number of iterations for the Markov chain to converge to the equilibrium distribution).

To estimate the 68% interval to two digits of precision, at least 1e5 (or 100,000) samples are needed. For the 95% interval, 1e6 (or 1,000,000) samples are needed. 1e4 samples gives a 'quick-and-dirty' approximation of the uncertainty. 

In [ ]:
result = fitter.fit_bayesian(samples=10000, burn=100)
fitter.plot_results(
    show_residuals=True, log_scale=True
)  # Plotting the fit and residuals is the same as before.

In [ ]:
# The plots don't show up in the jupyterbook, so we wrap them in a widget here.
# They do show up when students run the notebook, so we remove that cell in the student notebooks
from plotly.graph_objects import FigureWidget

FigureWidget(fitter.plot_results(show_residuals=True, log_scale=True))

We can now look at the fitting output in detail using a variety of plots:


i. A 'corner' plot showing a grid of parameter distribution from the Bayesian multi-parameter analysis.

ii. A marginal posterior plot showing the probability distribution for a single, chosen parameter.

iii. A Bayesian posterior predictive 95% credible band plot, which displays the model's predicted outcomes over a range of inputs (shaded region may be invisible depending on constraints).

iv. A parameter heatmap giving a colour-coded grid of relationships and statistical dependencies between model parameters estimated from Bayesian inference.

v. The Markov chain Monte Carlo (MCMC) trace which demonstrates how all of the parameters evolved during the fit.

First, the corner plot. Each panel on the diagonal shows the marginal posterior distribution of one varied parameter, while the off-diagonal panels show the pairwise relationships between parameters. 

Look out for strongly tilted or curved shapes in the off-diagonal panels: these indicate correlated parameters, which the data cannot constrain independently of one another.

In [ ]:
print("\nGenerating posterior pair (corner) plot...")
fitter.plot_posterior_pairs()

In [ ]:
FigureWidget(fitter.plot_posterior_pairs())

Next, we can single out one parameter of particular interest — here the sphere radius — and inspect its marginal posterior distribution in detail. 

Is the distribution symmetric? Roughly Gaussian? The width of this distribution is a direct measure of the uncertainty in the radius.

In [ ]:
print("\nGenerating marginal posterior for radius...")
fitter.plot_param_distribution("radius")

In [ ]:
FigureWidget(fitter.plot_param_distribution("radius"))

The posterior predictive plot propagates the parameter uncertainties back into $I(q)$ space: the shaded band shows the 95% credible interval of the model prediction. If the parameters are tightly constrained, the band may be so narrow that it is barely visible behind the data.

In [ ]:
print("\nGenerating posterior predictive band...")
fitter.plot_posterior_predictive(style="band")

In [ ]:
FigureWidget(fitter.plot_posterior_predictive(style="band"))

The correlation heatmap condenses the off-diagonal information from the corner plot into a single colour-coded matrix of correlation coefficients. Values close to +1 or −1 flag pairs of parameters that the data cannot determine independently.

In [ ]:
print("\nGenerating parameter correlation heatmap...")
fitter.plot_param_correlations()

In [ ]:
FigureWidget(fitter.plot_param_correlations())

Finally, the MCMC trace shows how each parameter evolved during the sampling. A well-converged chain should look like stationary noise around a constant value; systematic drifts or sudden jumps suggest that more `burn` iterations or more `samples` are needed.

In [ ]:
print("\nGenerating MCMC trace plot...")
fitter.plot_trace()

In [ ]:
FigureWidget(fitter.plot_trace())

Before moving on, compare these results with the Nelder-Mead fit from the [previous notebook](./6a-analysis-sans.ipynb):

- Are the median parameter values consistent with the maximum likelihood estimates found earlier?
- What extra information does the Bayesian analysis provide that a simple optimiser does not?

Finally, an example of how one can access the posterior data and save the fit results:

In [ ]:
posterior = fitter.get_posterior()
print("\nSampled parameters:", posterior.labels)
print("Chain shape:", posterior.samples.shape)
print("95% credible intervals:")
for name in posterior.labels:
    low, high = posterior.ci_95[name]
    print(f"  {name}: [{low:.6g}, {high:.6g}]")

# Export the raw chain for external analysis (pandas, corner, arviz, ...)
posterior.save_posterior_csv("posterior_chain.csv")
print("\n✓ Raw posterior chain saved to posterior_chain.csv")

# The saved fit results include the credible intervals in the header
fitter.save_results("bayesian_fit_results.csv")

### Exercise 7: Explore models for the data

You are now armed with knowledge of how to set models, variable and fixed parameters and constraints, and to then fit data with straightforward and more statistically rigorous approaches.

In the [previous notebook](./6a-analysis-sans.ipynb) we explored this dataset using spherical and ellipsoidal fits. Can Bayesian analysis help you to distinguish between candidate models?

In this exercise, you should:

1. Fetch the list of available models; documentation on these `SasView` models can be found [here](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/models/index.html).
2. Pick one, or several, models that could plausibly describe the data.
3. For each model: create a fitter, set initial values, bounds, and which parameters vary, then run a Bayesian fit with `fit_bayesian`.
4. Inspect the fit, residuals, and posterior distributions using the plotting methods from Exercise 6, and decide which model gives the most convincing description of the data.

**Further work:**

You could also explore the effects of limiting the fitted q-range to see how this limitation affects the statistical distributions:

```Python
fitter.set_q_range(qmin=0.1, qmax=0.3)
```

or try adding a simple structure factor:

```Python
fitter.set_structure_factor('hardsphere', radius_effective_mode='link_radius')
```

In [ ]:
from sans_fitter import get_all_models

print(get_all_models())

Use the empty cells below as your workspace. The workflow is the same as for the sphere in Exercise 6: `set_model`, `set_param`, `fit_bayesian`, and then the plotting methods introduced above.

**Solution:**

As one example, we can try a cylinder model. A cylinder has two size parameters — a radius and a length — so, as for the ellipsoid in the previous notebook, we let both vary alongside the scale and background:

In [ ]:
fitteri = SANSFitter()
fitteri.load_data(filename)
fitteri.set_model("cylinder")
fitteri.get_params()

In [ ]:
fitteri.set_param("sld", value=3, min=1, max=30, vary=False)
fitteri.set_param("sld_solvent", value=6, min=1, max=30, vary=False)

fitteri.set_param("radius", value=80, min=10, max=300, vary=True)
fitteri.set_param("length", value=80, min=10, max=300, vary=True)

fitteri.set_param("scale", value=1.4e-7, min=0, max=1, vary=True)
fitteri.set_param("background", value=0.1, min=0, max=1, vary=True)

fitteri.get_params()

In [ ]:
result = fitteri.fit_bayesian(samples=10000, burn=100)
fitteri.plot_results(show_residuals=True, log_scale=True)

In [ ]:
FigureWidget(fitteri.plot_results(show_residuals=True, log_scale=True))

In [ ]:
print("\nGenerating posterior pair (corner) plot...")
fitteri.plot_posterior_pairs()

In [ ]:
FigureWidget(fitteri.plot_posterior_pairs())

We can compare this with a second candidate: a vesicle, i.e. a hollow spherical shell with a solvent-filled core, characterised by a core radius and a membrane thickness. Note that for the vesicle model the `scale` parameter is fixed at 1, and the particle volume fraction `volfraction` is varied instead:

In [ ]:
fitteri = SANSFitter()
fitteri.load_data(filename)
fitteri.set_model("vesicle")
fitteri.get_params()

In [ ]:
fitteri.set_param("sld", value=3, min=1, max=30, vary=False)
fitteri.set_param("sld_solvent", value=6, min=1, max=30, vary=False)

fitteri.set_param("radius", value=80, min=10, max=300, vary=True)
fitteri.set_param("thickness", value=10, min=5, max=100, vary=True)
fitteri.set_param("volfraction", value=0.1, min=0, max=1, vary=True)


fitteri.set_param("scale", value=1, min=0, max=1, vary=False)
fitteri.set_param("background", value=0.1, min=0, max=1, vary=True)

fitteri.get_params()

In [ ]:
result = fitteri.fit_bayesian(samples=10000, burn=100)
fitteri.plot_results(show_residuals=True, log_scale=True)

In [ ]:
FigureWidget(fitteri.plot_results(show_residuals=True, log_scale=True))

In [ ]:
print("\nGenerating posterior pair (corner) plot...")
fitteri.plot_posterior_pairs()

In [ ]:
FigureWidget(fitteri.plot_posterior_pairs())

In [ ]:
print("\nGenerating posterior predictive band...")
fitteri.plot_posterior_predictive(style="band")

In [ ]:
FigureWidget(fitteri.plot_posterior_predictive(style="band"))

How do the different models compare? Do the fits, residuals, and posterior distributions single out one model as the best description of the data?

Consider also the physical plausibility of the fitted parameters: a model that fits well but requires unphysical parameter values should still be treated with suspicion.